# HW5: Monitoring

LLM Zoomcamp 2026 — Module 5: OpenTelemetry instrumentation for RAG

## Setup

In [ ]:
import os
import sqlite3

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult

load_dotenv()

from gitsource import GithubRepositoryDataReader
from minsearch import Index
from rag_helper import RAGBase

In [ ]:
# Load 72 course lessons (same as hw1, hw2, hw4)
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} documents")

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

# Using Groq (OpenAI-compatible) — no OpenAI key required
client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

## RAGTraced subclass

In [ ]:
class RAGTraced(RAGBase):
    """Wraps rag(), search(), llm() each in their own OTel span."""

    def __init__(self, tracer, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = tracer

    def llm(self, prompt):
        """Override to use chat completions (Groq-compatible)."""
        response = self.llm_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": self.instructions},
                {"role": "user", "content": prompt}
            ]
        )
        return response

    def rag(self, query):
        with self.tracer.start_as_current_span("rag") as rag_span:
            with self.tracer.start_as_current_span("search"):
                search_results = self.search(query)

            prompt = self.build_prompt(query, search_results)

            with self.tracer.start_as_current_span("llm") as llm_span:
                response = self.llm(prompt)
                usage = response.usage
                input_tok = usage.prompt_tokens
                output_tok = usage.completion_tokens
                llm_span.set_attribute("input_tokens", input_tok)
                llm_span.set_attribute("output_tokens", output_tok)
                cost = (input_tok * 0.59 + output_tok * 0.79) / 1_000_000
                llm_span.set_attribute("cost", cost)

            answer = response.choices[0].message.content
            rag_span.set_attribute("question", query)
            return answer

## Q1. First trace — counting spans

In [ ]:
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")

rag = RAGTraced(tracer=tracer, index=index, llm_client=client)

QUERY = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(QUERY)
print(answer)

In [ ]:
# Count the spans printed above
# Each ReadableSpan entry = one span
# Q1 Answer: 3 (search, llm, rag)
print("Q1: 3 spans produced (rag, search, llm)")

## SQLiteSpanExporter

In [ ]:
class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

## Q2–Q6: Run 4 times, query SQLite

In [ ]:
DB_PATH = "traces.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

sqlite_exp = SQLiteSpanExporter(DB_PATH)
provider2 = TracerProvider()
provider2.add_span_processor(SimpleSpanProcessor(sqlite_exp))
# Note: can't override global tracer provider after first set_tracer_provider()
# Use provider2 directly to get a tracer
tracer2 = provider2.get_tracer("llm-zoomcamp-sqlite")

rag2 = RAGTraced(tracer=tracer2, index=index, llm_client=client)

# Run 4 times (Q6 requires 4 runs)
for i in range(4):
    _ = rag2.rag(QUERY)
    print(f"Run {i+1} complete")

In [ ]:
df = pd.read_sql(
    "SELECT name, start_time, end_time, input_tokens, output_tokens, cost FROM spans",
    sqlite_exp.conn
)
df["duration_ms"] = (df["end_time"] - df["start_time"]) / 1_000_000
df

In [ ]:
# Q2: input tokens for LLM call
first_llm = df[df["name"] == "llm"].iloc[0]
print(f"Q2: input_tokens = {int(first_llm['input_tokens'])}")
# → 7206 → select 7000

In [ ]:
# Q3: LLM call duration
print(f"Q3: LLM duration = {first_llm['duration_ms']:.0f}ms")
# → ~1023ms → select 500-2000ms

In [ ]:
# Q4: span names in the table
print(f"Q4: span names = {df['name'].unique().tolist()}")
# → ['search', 'llm', 'rag'] → select 'rag, search, and llm'

In [ ]:
# Q5: which span takes most time (excluding rag)
non_rag = df[df["name"] != "rag"].groupby("name")["duration_ms"].sum()
print(non_rag)
print(f"Q5: slowest = {non_rag.idxmax()}")
# → llm (86541ms vs search 7ms)

In [ ]:
# Q6: input token stability across 4 runs
llm_tokens = df[df["name"] == "llm"]["input_tokens"].tolist()
print(f"Token counts: {llm_tokens}")
diff = max(llm_tokens) - min(llm_tokens)
print(f"Max diff: {diff}")
# → [7206, 7206, 7206, 7206] → They are identical

## Summary

| Q | Result | Answer |
|---|--------|--------|
| Q1 | 3 spans per trace | **3** |
| Q2 | 7206 input tokens | **7000** |
| Q3 | ~1023ms | **500–2000ms** |
| Q4 | rag, search, llm | **rag, search, and llm** |
| Q5 | llm: 86541ms vs search: 7ms | **llm** |
| Q6 | [7206, 7206, 7206, 7206] | **They are identical** |